# Kernels

In [ ]:
import gpytorch
import math
import torch
from matplotlib import pyplot as plt
import numpy as np

# %matplotlib inline
import plotly.graph_objects as go

In [ ]:
# Training data is 100 points in [0,1] inclusive regularly spaced
train_x = torch.linspace(0, 1, 0)

# True function is sin(2*pi*x) with Gaussian noise
train_y = torch.sin(train_x * (2 * math.pi)) + torch.randn(train_x.size()) * math.sqrt(0.04)

In [ ]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(train_x, train_y, likelihood)

In [ ]:
model.covar_module.base_kernel.parameters()

In [ ]:
# this is for running the notebook in our testing framework
import os
smoke_test = ('CI' in os.environ)
training_iter = 2 if smoke_test else 50


# Find optimal model hyperparameters
model.train()
likelihood.train()

# Use the adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1)  # Includes GaussianLikelihood parameters

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iter):
    # Zero gradients from previous iteration
    optimizer.zero_grad()
    # Output from model
    output = model(train_x)
    # Calc loss and backprop gradients
    loss = - mll(output, train_y)
    loss.backward()
    print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
        i + 1, training_iter, loss.item(),
        model.covar_module.base_kernel.lengthscale.item(),
        model.likelihood.noise.item()
    ))
    optimizer.step()

In [ ]:
# Get into evaluation (predictive posterior) mode
model.eval()
likelihood.eval()

# Test points are regularly spaced along [0,1]
# Make predictions by feeding model through likelihood
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    test_x = torch.linspace(0, 2, 51)
    f_preds = model(test_x)
    y_preds = likelihood(model(test_x))

In [ ]:
f_mean = f_preds.mean
f_var = f_preds.variance
f_covar = f_preds.covariance_matrix

In [ ]:
plt.imshow(f_covar.detach().numpy(), cmap = 'viridis')
plt.title("Posterior covariance matrix across larger domain")
# legend
plt.colorbar()
plt.show()

In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 1, figsize = (8, 6))

    # Get upper and lower confidence bounds
    lower, upper = y_preds.confidence_region()
    # Plot training data as black stars
    ax.plot(train_x.numpy(), train_y.numpy(), 'k*')
    # Plot predictive means as blue line
    ax.plot(test_x.numpy(), y_preds.mean.numpy(), 'b')
    # Shade between the lower and upper confidence bounds
    ax.fill_between(test_x.numpy(), lower.numpy(), upper.numpy(), alpha = 0.5)
    ax.set_ylim([-3, 3])
    ax.legend(['Observed Data', 'Mean', 'Confidence'])

# 2 D

In [ ]:
n_train = 100

# Training data is 100 points in [0,1] inclusive regularly spaced
train_x = torch.linspace(0, 1, n_train)

mesh1, mesh2 = np.meshgrid(train_x, train_x)
# mesh == mesh2.T

In [ ]:
train_x_expldim = train_x.unsqueeze(0)
train_x_2D = torch.cat((train_x_expldim, train_x_expldim), dim = 0)
train_x_2D.shape

In [ ]:
tile = torch.tile(train_x_expldim, dims = (100, 1))

noise = 0 # 0.04
# train_y = torch.sin((train_x_2D * 0.5) + (train_x_2D[0] + train_x_2D[1] * (math.pi))
#            + torch.randn(train_x_2D.size()[1]) * math.sqrt(noise))

train_y = (torch.sin(torch.tensor((mesh1 * 4) + (mesh2 * 4)) + (tile * 4)) * 0.25)

In [ ]:
 fig = go.Figure(data = [go.Surface(
    z = train_y, 
    x = train_x_2D[0], 
    y = train_x_2D[1],
    opacity = 0.7
    )])

fig.update_layout(title = 'Surface Mass Balance (SMB) for Antarctica - Dec 2022',
                  width = 700, height = 700,
                  margin = dict(l = 65, r = 50, b = 65, t = 90)
                  )

fig.show()